# 34 · Preparar el ejecutor de entrenamiento, sin entrenar

**Revisión 1.0.** Selecciona **CPU** y ejecuta las celdas en orden. Esta libreta verifica el código del ejecutor y su configuración, usando pruebas sintéticas y la comprobación de modelos que ya guardaste con la libreta 33.

Al pedir la carpeta de Drive, pega **la misma carpeta de registros privados donde guardaste la comprobación 33**. La libreta busca únicamente sus archivos JSON dentro de las subcarpetas directas; selecciona el registro completo y válido más reciente. No hace falta volver a ejecutar la 33 ni buscar su subcarpeta manualmente.

No se descargarán pesos, no se abrirán paquetes de participantes y no hay ninguna celda que inicie entrenamiento. La prueba corta en GPU y el entrenamiento completo corresponden a una etapa posterior, con autorización propia. Todo el código queda en el repositorio; Drive conserva los resultados de esta comprobación en una carpeta nueva.

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = 'PENDING_PUBLISHED_COMMIT'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') == '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona CPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_training_preparation34_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import training_preparation_colab as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La comprobación se detuvo. No ejecutes entrenamiento.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')


## 1. Preparar el intérprete CPU aislado

Se conservan las versiones verificadas de las bibliotecas y se añaden las dependencias clínicas fijadas del ejecutor. La preparación puede tardar varios minutos; no altera el intérprete principal de Colab.

In [ ]:
print('Preparando el intérprete CPU aislado del ejecutor…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/check_training_preparation_colab.py'), '--operation', 'setup', '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'cpu_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR)
print('Entorno CPU aislado preparado. No se descargaron pesos ni se abrieron imágenes.')


## 2. Ejecutar las pruebas sintéticas del código

Se verifican las reglas de autorización, la separación por participante, las entradas del ejecutor, los modelos y el guardado de los registros con datos sintéticos. No se ejecutan pasos de entrenamiento, optimización ni cálculo de gradientes. Cualquier error o prueba omitida detiene la preparación.

In [ ]:
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_training_preparation_colab.py'), '--operation', 'tests', '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=RUN_ENV))
print(f"Pruebas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Localizar la comprobación aprobada de la libreta 33

Monta tu Drive y pega la carpeta privada de registros donde guardaste el resultado anterior. Solo se leerán los dos JSON de esa comprobación; no selecciones una carpeta de imágenes ni el paquete reservado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_ROOT = orchestration.checked_path(input('Carpeta existente de KneeOAI donde guardaste la comprobación de la libreta 33: ').strip())
if not PERSISTENT_ROOT.is_dir() or not PERSISTENT_ROOT.is_relative_to(Path('/content/drive')):
    raise RuntimeError('Selecciona una carpeta existente dentro del Drive montado.')
previous_record = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_training_preparation_colab.py'), '--operation', 'locate', '--repository-root', str(REPO_DIR), '--records-root', str(PERSISTENT_ROOT)], cwd=REPO_DIR, env=RUN_ENV))
SOURCE_PREFLIGHT = Path(previous_record['source_preflight_dir'])
print('Comprobación completa de la libreta 33 localizada y verificada.')


## 4. Verificar que el ejecutor queda preparado

Se comprueba el registro de los modelos, la configuración aprobada y las barreras que exigirán autorización antes de entrenar. La verificación usa únicamente los registros anteriores y entradas sintéticas. No genera un permiso para entrenar.

In [ ]:
import uuid
LOCAL_OUTPUT = SESSION_ROOT / ('preparation_' + uuid.uuid4().hex)
print('Verificando la preparación del ejecutor. Sin entrenamiento…')
preparation_log = checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_training_preparation_colab.py'), '--operation', 'prepare', '--repository-root', str(REPO_DIR), '--source-preflight-dir', str(SOURCE_PREFLIGHT), '--output-dir', str(LOCAL_OUTPUT), '--execution-environment', 'google_colab_cpu'], cwd=REPO_DIR, env=RUN_ENV)
result = json.loads((LOCAL_OUTPUT / 'training_runner_preparation.json').read_text(encoding='utf-8'))
orchestration.validate_result(result, 'google_colab_cpu')
print('Ejecutor preparado; prueba corta en GPU y entrenamiento pendientes de autorización.')


## 5. Guardar la comprobación en la misma carpeta de Drive

Se crean dos archivos JSON en una subcarpeta nueva junto a los registros anteriores. No se reemplaza ningún resultado previo.

In [ ]:
FINAL_CHECK = orchestration.publish_result(LOCAL_OUTPUT, PERSISTENT_ROOT, CODE_REVISION, CPU_TESTS, runtime)
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('Preparación guardada. No se entrenó ni se abrió la prueba reservada.')


Comparte únicamente el **JSON final**. No necesitas enviar imágenes ni paquetes privados. Que esta preparación pase no significa que ya se haya ejecutado la prueba corta en GPU o autorizado el entrenamiento.